In [ ]:
import geopandas as gpd
import rasterio
from pathlib import Path

RAW_PORCINO   = Path("../../data/raw/01_ganado_porcino")
RAW_GAS       = Path("../../data/raw/02_gasoductos")
RAW_PENDIENTE = Path("../../data/raw/03_pendiente_dem")
RAW_NATURA    = Path("../../data/raw/04_red_natura2000")
RAW_VIARIA    = Path("../../data/raw/05_red_viaria")
RAW_SUELO     = Path("../../data/raw/06_clasificacion_suelo")
DELIM_DIR     = Path("../../data/raw/delimitations")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

gaseoductos = gpd.read_file(RAW_GAS / "gasoductos_huesca.gpkg")
clasificacion_suelo = gpd.read_file(RAW_SUELO / "clasificacion_suelo_huesca.gpkg")
densidad_porcino = gpd.read_file(RAW_PORCINO / "clasificacion_porcino.gpkg")
pendiente = rasterio.open(RAW_PENDIENTE / "pendiente_huesca_provincia.tif")  # nombre real (antes decía pendiente_huesca_25m.tif, que no existe)
carreteras = gpd.read_file(RAW_VIARIA / "carreteras_camiones_huesca.gpkg", layer="edges").to_crs("EPSG:25830")
bufers_nucleos = gpd.read_file(RAW_SUELO / "buffer_nucleos_urbanos.gpkg")  # ⚠️ pendiente: 06_clasificacion_suelo aún no exporta este archivo

print(gaseoductos.head(2))
print(clasificacion_suelo.head(2))
print(densidad_porcino.head(2))
print(pendiente.meta)
print(carreteras.head(2))
print(bufers_nucleos.head(2))

print(gaseoductos.crs)
print(clasificacion_suelo.crs)
print(densidad_porcino.crs)
print(pendiente.crs)
print(carreteras.crs)
print(bufers_nucleos.crs)

print(gaseoductos.shape)
print(clasificacion_suelo.shape)
print(densidad_porcino.shape)
print(pendiente.width, pendiente.height)
print(carreteras.shape)
print(bufers_nucleos.shape)


In [ ]:
carreteras = carreteras.to_crs(25830)
print(carreteras.crs)  # confirma EPSG:25830

EPSG:25830


In [ ]:
import geopandas as gpd
from pathlib import Path

DELIM_DIR = Path("../../data/raw/delimitations")

# Cargar la delimitación
huesca = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson")

# 1. CRS y proyección
print("=== CRS ===")
print(huesca.crs)
print("¿Está proyectado (metros)?", huesca.crs.is_projected)

# 2. Geometría válida
print("\n=== GEOMETRÍA ===")
print("Número de features:", len(huesca))
print("Tipos de geometría:", huesca.geom_type.unique())
print("Geometrías válidas:", huesca.is_valid.all())
print("Geometrías nulas:", huesca.geometry.isnull().sum())

# 3. Extensión
xmin, ymin, xmax, ymax = huesca.total_bounds
print("\n=== EXTENSIÓN (metros) ===")
print(f"X: {xmin:.0f} - {xmax:.0f}  →  ancho: {(xmax-xmin)/1000:.1f} km")
print(f"Y: {ymin:.0f} - {ymax:.0f}  →  alto:  {(ymax-ymin)/1000:.1f} km")

# 4. Estimación del número de celdas
cell_size = 500
n_cols = int((xmax - xmin) / cell_size)
n_rows = int((ymax - ymin) / cell_size)
print(f"\n=== ESTIMACIÓN GRID 500m ===")
print(f"Celdas brutas (bbox completa): {n_cols * n_rows:,}")
print(f"Celdas aprox. tras clip (~60-70% bbox): {int(n_cols * n_rows * 0.65):,}")

In [ ]:
import geopandas as gpd
from pathlib import Path

DELIM_DIR = Path("../../data/raw/delimitations")

# Cargar y reproyectar
huesca = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson")
huesca = huesca.to_crs("EPSG:25830")

# Verificar tras reproyección
xmin, ymin, xmax, ymax = huesca.total_bounds
print("CRS:", huesca.crs)
print(f"X: {xmin:.0f} - {xmax:.0f}  →  ancho: {(xmax-xmin)/1000:.1f} km")
print(f"Y: {ymin:.0f} - {ymax:.0f}  →  alto:  {(ymax-ymin)/1000:.1f} km")

cell_size = 500
n_cols = int((xmax - xmin) / cell_size)
n_rows = int((ymax - ymin) / cell_size)
print(f"\nCeldas brutas (bbox completa): {n_cols * n_rows:,}")
print(f"Celdas aprox. tras clip (~60-70% bbox): {int(n_cols * n_rows * 0.65):,}")

In [ ]:
import geopandas as gpd
import numpy as np
from shapely.geometry import box
from pathlib import Path

DELIM_DIR = Path("../../data/raw/delimitations")
PROCESSED_DIR = Path("../../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Cargar y reproyectar
huesca = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:25830")

xmin, ymin, xmax, ymax = huesca.total_bounds
cell_size = 500

# Generar grid completo (bbox)
x_coords = np.arange(xmin, xmax, cell_size)
y_coords = np.arange(ymin, ymax, cell_size)

cells = [
    box(x, y, x + cell_size, y + cell_size)
    for x in x_coords
    for y in y_coords
]

grid = gpd.GeoDataFrame(geometry=cells, crs="EPSG:25830")
print(f"Celdas brutas generadas: {len(grid):,}")

# Paso 1: filtro rápido con sjoin (descarta celdas claramente fuera)
grid_filtrado = gpd.sjoin(grid, huesca[["geometry"]], how="inner", predicate="intersects")
grid_filtrado = grid_filtrado.drop(columns="index_right").drop_duplicates()
print(f"Celdas tras sjoin: {len(grid_filtrado):,}")

# Paso 2: clip preciso con el límite real
grid_final = gpd.clip(grid_filtrado, huesca)
grid_final = grid_final.reset_index(drop=True)
grid_final["cell_id"] = range(len(grid_final))

print(f"Celdas finales tras clip: {len(grid_final):,}")
print(grid_final.head())

# Guardar
grid_final.to_file(PROCESSED_DIR / "grid_huesca_500m.gpkg", driver="GPKG")
print(f"\nGrid guardado: {PROCESSED_DIR / 'grid_huesca_500m.gpkg'}")

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_GAS = Path("../../data/raw/02_gasoductos")

grid = gpd.read_file(GRID_PATH)

if "dist_gasoducto" not in grid.columns:
    gasoductos = gpd.read_file(RAW_GAS / "gasoductos_huesca.gpkg").to_crs("EPSG:25830")
    gasoductos_union = unary_union(gasoductos.geometry)
    grid["centroid"] = grid.geometry.centroid
    grid["dist_gasoducto"] = grid["centroid"].apply(lambda p: p.distance(gasoductos_union))
    grid = grid.drop(columns="centroid")
    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: dist_gasoducto")
else:
    print("dist_gasoducto ya existe, saltando cálculo")

print("Min:", round(grid["dist_gasoducto"].min(), 1), "m")
print("Max:", round(grid["dist_gasoducto"].max() / 1000, 1), "km")
print("Media:", round(grid["dist_gasoducto"].mean() / 1000, 1), "km")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_VIARIA = Path("../../data/raw/05_red_viaria")

grid = gpd.read_file(GRID_PATH)

if "dist_carretera" not in grid.columns:
    carreteras = gpd.read_file(RAW_VIARIA / "carreteras_camiones_huesca.gpkg").to_crs("EPSG:25830")

    centroides = gpd.GeoDataFrame(
        {"cell_id": grid["cell_id"]},
        geometry=grid.geometry.centroid,
        crs="EPSG:25830"
    )

    resultado = gpd.sjoin_nearest(
        centroides,
        carreteras[["geometry"]],
        how="left",
        distance_col="dist_carretera"
    )
    resultado = resultado[~resultado.index.duplicated(keep="first")].sort_index()
    grid["dist_carretera"] = resultado["dist_carretera"].values

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: dist_carretera")
else:
    print("dist_carretera ya existe, saltando cálculo")

print("Min:", round(grid["dist_carretera"].min(), 1), "m")
print("Max:", round(grid["dist_carretera"].max() / 1000, 1), "km")
print("Media:", round(grid["dist_carretera"].mean() / 1000, 1), "km")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_PORCINO = Path("../../data/raw/01_ganado_porcino")

grid = gpd.read_file(GRID_PATH)

if "biomasa_10km" not in grid.columns:
    densidad_porcino = gpd.read_file(RAW_PORCINO / "clasificacion_porcino.gpkg").to_crs("EPSG:25830")
    
    centroides_buf = grid.copy()
    centroides_buf["geometry"] = grid.geometry.centroid.buffer(10_000)

    joined = gpd.sjoin(
        densidad_porcino[["capacidad_total", "geometry"]],
        centroides_buf[["cell_id", "geometry"]].set_geometry("geometry"),
        how="left",
        predicate="within"
    )

    biomasa = joined.groupby("cell_id")["capacidad_total"].sum().reset_index()
    biomasa.columns = ["cell_id", "biomasa_10km"]

    grid = grid.merge(biomasa, on="cell_id", how="left")
    grid["biomasa_10km"] = grid["biomasa_10km"].fillna(0)

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: biomasa_10km")
else:
    print("biomasa_10km ya existe, saltando cálculo")

print(grid["biomasa_10km"].describe().round(0))

In [ ]:
import geopandas as gpd
import ast
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_VIARIA = Path("../../data/raw/05_red_viaria")

grid = gpd.read_file(GRID_PATH)

if "pts_carretera" not in grid.columns:
    carreteras = gpd.read_file(RAW_VIARIA / "carreteras_camiones_huesca.gpkg").to_crs("EPSG:25830")

    puntuacion_highway = {
        "motorway":       4,
        "trunk":          4,
        "primary":        3,
        "secondary":      2,
        "tertiary":       1,
        "motorway_link":  3,
        "trunk_link":     3,
        "primary_link":   2,
        "secondary_link": 1,
        "tertiary_link":  1,
    }

    def puntuar_highway(valor):
        if isinstance(valor, list):
            tipos = valor
        elif isinstance(valor, str) and valor.startswith("["):
            try:
                tipos = ast.literal_eval(valor)
            except:
                tipos = [valor]
        else:
            tipos = [valor]
        puntos = [puntuacion_highway.get(t, 0) for t in tipos]
        return max(puntos) if puntos else 0

    carreteras["pts_highway"] = carreteras["highway"].apply(puntuar_highway)

    centroides = gpd.GeoDataFrame(
        {"cell_id": grid["cell_id"]},
        geometry=grid.geometry.centroid,
        crs="EPSG:25830"
    )

    resultado = gpd.sjoin_nearest(
        centroides,
        carreteras[["pts_highway", "geometry"]],
        how="left",
        distance_col="dist_carretera_check"
    )
    resultado = resultado.sort_values("pts_highway", ascending=False)
    resultado = resultado[~resultado.index.duplicated(keep="first")].sort_index()

    grid["pts_carretera"] = resultado["pts_highway"].values

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: pts_carretera")
else:
    print("pts_carretera ya existe, saltando cálculo")

print(grid["pts_carretera"].value_counts().sort_index())

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_SUELO = Path("../../data/raw/06_clasificacion_suelo")

grid = gpd.read_file(GRID_PATH)

if "dist_nucleos" not in grid.columns:
    buffers = gpd.read_file(RAW_SUELO / "buffer_nucleos_urbanos.gpkg").to_crs("EPSG:25830")  # ⚠️ ver nota en celda 1: falta exportarlo desde 06_clasificacion_suelo
    buffers_union = unary_union(buffers.geometry)

    def distancia_con_signo(punto, poligono):
        dist_borde = poligono.exterior.distance(punto) if hasattr(poligono, "exterior") else poligono.boundary.distance(punto)
        if poligono.contains(punto):
            return -dist_borde
        return dist_borde

    grid["centroid"] = grid.geometry.centroid
    grid["dist_nucleos"] = grid["centroid"].apply(lambda p: distancia_con_signo(p, buffers_union))
    grid = grid.drop(columns="centroid")

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: dist_nucleos")
else:
    print("dist_nucleos ya existe, saltando cálculo")

print("Fuera del buffer (dist > 0):", (grid["dist_nucleos"] > 0).sum())
print("Dentro del buffer (dist < 0):", (grid["dist_nucleos"] < 0).sum())
print("Min:", round(grid["dist_nucleos"].min(), 1), "m")
print("Max:", round(grid["dist_nucleos"].max() / 1000, 1), "km")

In [ ]:
import geopandas as gpd
from pathlib import Path

RAW_SUELO = Path("../../data/raw/06_clasificacion_suelo")

buffers = gpd.read_file(RAW_SUELO / "buffer_nucleos_urbanos.gpkg").to_crs("EPSG:25830")  # ⚠️ ver nota en celda 1

print("Número de buffers:", len(buffers))
print("Columnas:", buffers.columns.tolist())
print("\nPrimeras filas:")
print(buffers.head(10))

# Área de cada buffer
buffers["area_km2"] = buffers.geometry.area / 1_000_000
print("\nEstadísticas de área (km²):")
print(buffers["area_km2"].describe())

# Cobertura total sobre Huesca (~15.600 km²)
cobertura = buffers.geometry.union_all().area / 1_000_000
print(f"\nCobertura total de buffers: {cobertura:.1f} km²")
print(f"Sobre el total de Huesca (~15.600 km²): {cobertura/15600*100:.1f}%")

In [ ]:
import geopandas as gpd
from rasterstats import zonal_stats
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_PENDIENTE = Path("../../data/raw/03_pendiente_dem")

grid = gpd.read_file(GRID_PATH)

if "pendiente_media" not in grid.columns:
    stats = zonal_stats(
        grid,
        str(RAW_PENDIENTE / "pendiente_huesca_provincia.tif"),  # nombre real (antes decía pendiente_huesca_25m.tif)
        stats=["mean", "max"],
        nodata=-9999,
        all_touched=False
    )

    grid["pendiente_media"] = [s["mean"] for s in stats]
    grid["pendiente_max"]   = [s["max"]  for s in stats]

    # Imputar nulos con la media global
    grid["pendiente_media"] = grid["pendiente_media"].fillna(grid["pendiente_media"].mean())
    grid["pendiente_max"] = grid["pendiente_max"].fillna(grid["pendiente_max"].mean())

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columnas: pendiente_media, pendiente_max")
else:
    print("pendiente_media ya existe, saltando cálculo")

print("Min:", round(grid["pendiente_media"].min(), 2), "°")
print("Max:", round(grid["pendiente_media"].max(), 2), "°")
print("Media:", round(grid["pendiente_media"].mean(), 2), "°")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_SUELO = Path("../../data/raw/06_clasificacion_suelo")

grid = gpd.read_file(GRID_PATH)

if "pts_suelo" not in grid.columns:
    suelo = gpd.read_file(RAW_SUELO / "clasificacion_suelo_huesca.gpkg").to_crs("EPSG:25830")

    puntuacion_clase = {
        "SENU":  3,
        "SDUD":  2,
        "SEUND": 2,
        "SDUNC": 2,
        "SENE":  0,
        "SUC":   0,
        "SUNC":  0,
    }

    puntuacion_uso = {
        "I":        2,
        "INF":      1,
        "EA":       1,
        "CA":       1,
        "RI":       1,
        "OD":       0,
        "T":        0,
        "GENERICO": 0,
        "ZB":       0,
        "R":       -1,
        "UR":      -1,
        "EC":      -1,
        "EQ":      -1,
        "EL":      -1,
        "EN":      -1,
        "RN":      -1,
        "Pa":      -1,
        "DF":      -1,
        "SU":      -1,
        "NO":      -1,
        "VP":      -1,
        "SGUR":    -1,
        "SGEQ":    -1,
        "SGEL":    -1,
        "SGINF":    0,
    }

    suelo["pts_clase"] = suelo["clase"].map(puntuacion_clase).fillna(0)
    suelo["pts_uso"]   = suelo["uso_glob"].map(puntuacion_uso).fillna(0)
    suelo["pts_suelo"] = suelo["pts_clase"] + suelo["pts_uso"]
    suelo.loc[suelo["pts_clase"] == 0, "pts_suelo"] = 0

    suelo_grid = gpd.overlay(grid[["cell_id", "geometry"]],
                             suelo[["pts_suelo", "geometry"]],
                             how="intersection")
    suelo_grid["area"] = suelo_grid.geometry.area

    idx_max = suelo_grid.groupby("cell_id")["area"].idxmax()
    dominante = suelo_grid.loc[idx_max, ["cell_id", "pts_suelo"]].set_index("cell_id")

    grid["pts_suelo"] = grid["cell_id"].map(dominante["pts_suelo"])
    grid["pts_suelo"] = grid["pts_suelo"].fillna(2)

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: pts_suelo")
else:
    print("pts_suelo ya existe, saltando cálculo")

print(grid["pts_suelo"].value_counts().sort_index())

In [ ]:
import geopandas as gpd
from shapely.ops import unary_union
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

RAW_NATURA = Path("../../data/raw/04_red_natura2000")

grid = gpd.read_file(GRID_PATH)

if "red_natura" not in grid.columns:
    natura = gpd.read_file(RAW_NATURA / "red_natura_huesca.gpkg").to_crs("EPSG:25830")

    zec   = natura[natura["SITETYPE"] == "B"]
    zepa  = natura[natura["SITETYPE"] == "A"]
    ambas = natura[natura["SITETYPE"] == "C"]

    geom_zec   = unary_union(zec.geometry)
    geom_zepa  = unary_union(zepa.geometry)
    geom_ambas = unary_union(ambas.geometry) if len(ambas) > 0 else None

    solapamiento = geom_zec.intersection(geom_zepa)
    ambas_final  = solapamiento.union(geom_ambas) if (geom_ambas and not geom_ambas.is_empty) else solapamiento

    centroides = grid.geometry.centroid

    def penalizacion_natura(punto):
        if ambas_final and not ambas_final.is_empty and ambas_final.contains(punto):
            return -5
        if not geom_zepa.is_empty and geom_zepa.contains(punto):
            return -3
        if not geom_zec.is_empty and geom_zec.contains(punto):
            return -3
        return 0

    grid["red_natura"] = centroides.apply(penalizacion_natura)

    grid.to_file(GRID_PATH, driver="GPKG")
    print("Calculado y guardado OK — columna: red_natura")
else:
    print("red_natura ya existe, saltando cálculo")

print(grid["red_natura"].value_counts().sort_index())

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

cols = ["dist_gasoducto", "biomasa_10km", "dist_nucleos", 
        "pts_carretera", "pendiente_media", "pendiente_max", 
        "pts_suelo", "red_natura"]

print("=== NULOS ===")
print(grid[cols].isnull().sum())

print("\n=== ESTADÍSTICAS ===")
print(grid[cols].describe().round(2))

print("\n=== DISTRIBUCIÓN pts_suelo ===")
print(grid["pts_suelo"].value_counts(dropna=False).sort_index())

print("\n=== DISTRIBUCIÓN pts_carretera ===")
print(grid["pts_carretera"].value_counts(dropna=False).sort_index())

print(f"\nTotal celdas: {len(grid)}")

In [ ]:
import geopandas as gpd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

# ── 1. Pendiente: imputar con media global (45 celdas, irrelevante) ──
media_pendiente     = grid["pendiente_media"].mean()
media_pendiente_max = grid["pendiente_max"].mean()

grid["pendiente_media"] = grid["pendiente_media"].fillna(media_pendiente)
grid["pendiente_max"] = grid["pendiente_max"].fillna(media_pendiente_max)

print(f"pendiente_media imputada con: {media_pendiente:.2f}°")
print(f"pendiente_max imputada con:   {media_pendiente_max:.2f}°")

# ── 2. pts_suelo: imputar nulos con 2 (condicionado) ────────────────
grid["pts_suelo"] = grid["pts_suelo"].fillna(2)
print(f"\npts_suelo nulos imputados con 2 (condicionado)")

# ── Verificación final ───────────────────────────────────────────────
cols = ["dist_gasoducto", "biomasa_10km", "dist_nucleos",
        "pts_carretera", "pendiente_media", "pendiente_max",
        "pts_suelo", "red_natura"]

print("\n=== NULOS TRAS LIMPIEZA ===")
print(grid[cols].isnull().sum())

grid.to_file(GRID_PATH, driver="GPKG")
print("\nGuardado OK — limpieza completada")

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

cols = ["dist_gasoducto", "biomasa_10km", "dist_nucleos", "dist_carretera",
        "pts_carretera", "pendiente_media", "pendiente_max",
        "pts_suelo", "red_natura",]

# VIF requiere matriz sin nulos (ya los limpiamos)
X = grid[cols].values

vif_data = pd.DataFrame()
vif_data["variable"] = cols
vif_data["VIF"] = [variance_inflation_factor(X, i) for i in range(X.shape[1])]
vif_data = vif_data.sort_values("VIF", ascending=False)

print("=== VIF ===")
print(vif_data.to_string(index=False))
print("\nReferencia: VIF > 10 → multicolinealidad alta, considerar eliminar")
print("            VIF 5-10 → moderada, vigilar")
print("            VIF < 5  → aceptable")

# Correlación entre variables también útil
print("\n=== CORRELACIÓN ===")
print(grid[cols].corr().round(2).to_string())

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

cols = ["dist_gasoducto", "biomasa_10km", "dist_nucleos", "dist_carretera", 
        "pts_carretera", "pendiente_media", "pendiente_max", "pts_suelo", "red_natura"]

# Renombrar para el gráfico
labels = {
    "dist_gasoducto": "Dist. Gasoducto",
    "biomasa_10km":   "Biomasa 10km",
    "dist_nucleos":   "Dist. Núcleos",
    "pendiente_media":"Pendiente",
    "pendiente_max":  "Pendiente Máxima",
    "pts_suelo":      "Suelo",
    "red_natura":     "Red Natura",
    "pts_carretera":  "Puntos Carretera",
    "dist_carretera": "Distancia Carretera",
    }

corr = grid[cols].rename(columns=labels).corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={"size": 10}
)
ax.set_title("Matriz de correlación — Variables del modelo", fontsize=13, pad=15)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "matriz_correlacion.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Guardada: {PROCESSED_DIR / 'matriz_correlacion.png'}")

In [ ]:
import geopandas as gpd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

def norm_minmax(serie):
    mn, mx = serie.min(), serie.max()
    return (serie - mn) / (mx - mn)

def norm_inv(serie):
    return 1 - norm_minmax(serie)

# dist_gasoducto: menos = mejor
OPTIMO_GAS = 3_000
MAXIMO_GAS = 10_000

grid["n_gasoducto"] = norm_inv(grid["dist_gasoducto"] ** 0.5)
# biomasa: más = mejor
grid["n_porcino"] = norm_minmax(grid["biomasa_10km"])

# dist_nucleos: función en U invertida
OPTIMO_MIN =   500
OPTIMO_MAX = 3_000

def score_nucleos(d):
    if d < 0:
        return 0.0
    elif d <= OPTIMO_MIN:
        return d / OPTIMO_MIN * 0.5
    elif d <= OPTIMO_MAX:
        return 1.0
    else:
        decay = 1.0 - (d - OPTIMO_MAX) / (20_000 - OPTIMO_MAX)
        return float(np.clip(decay, 0, 1))

grid["n_nucleos"] = grid["dist_nucleos"].apply(score_nucleos)

# pendiente: menos = mejor (cap 25°)
grid["n_pendiente"] = norm_inv(grid["pendiente_media"].clip(upper=25))

# dist_carretera: menos = mejor
grid["n_carretera_dist"] = norm_inv(grid["dist_carretera"])

# suelo: más = mejor
grid["n_suelo"] = norm_minmax(grid["pts_suelo"].astype(float))

# red_natura: binario (0 = sin protección → 1.0)
grid["n_natura"] = (grid["red_natura"] == 0).astype(float)
# ── pts_carretera: más puntuación = mejor ────────────────
grid["n_carretera"] = norm_minmax(grid["pts_carretera"].astype(float))
print(grid[["n_gasoducto", "n_porcino", "n_nucleos", "n_pendiente",
            "n_carretera_dist", "n_suelo", "n_natura"]].describe().round(3))

grid.to_file(GRID_PATH, driver="GPKG")
print("Guardado OK — columnas n_* calculadas")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

grid["motivo_exclusion"] = ""
grid.loc[grid["red_natura"] < 0,          "motivo_exclusion"] = "Red Natura"
grid.loc[grid["pts_suelo"] == 0,          "motivo_exclusion"] = "Suelo no apto"
grid.loc[grid["pendiente_media"] > 25,    "motivo_exclusion"] = "Pendiente excesiva"
grid.loc[grid["dist_nucleos"] < 0,        "motivo_exclusion"] = "Núcleo urbano"
grid.loc[grid["dist_gasoducto"] > 15_000, "motivo_exclusion"] = "Lejos del gasoducto"
grid.loc[grid["biomasa_10km"] < 20_000,   "motivo_exclusion"] = "Biomasa insuficiente"

mask_exclusion = grid["motivo_exclusion"] != ""
grid["categoria"] = "APTA"
grid.loc[mask_exclusion, "categoria"] = "EXCLUIDO"

print(f"Celdas aptas:     {(grid['categoria'] == 'APTA').sum():,}")
print(f"Celdas excluidas: {(grid['categoria'] == 'EXCLUIDO').sum():,}")

grid.to_file(GRID_PATH, driver="GPKG")
print("Guardado OK")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

COLS_MODELO = ["n_gasoducto", "n_porcino", "n_pendiente",
               "n_carretera_dist","n_carretera", "n_suelo", "n_nucleos"]

grid_aptas = grid[grid["categoria"] != "EXCLUIDO"].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(grid_aptas[COLS_MODELO].values)

inercias = []
ks = range(2, 16)
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(ks, inercias, "o-")
plt.xlabel("Número de clusters (k)")
plt.ylabel("Inercia")
plt.title("Elbow — K-Means sobre celdas aptas")
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "elbow_kmeans.png", dpi=120)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import pandas as pd
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

COLS_MODELO = ["n_gasoducto", "n_porcino", "n_pendiente",
               "n_carretera_dist","n_carretera", "n_suelo", "n_nucleos"]

grid_aptas = grid[grid["categoria"] != "EXCLUIDO"].copy()

X = grid_aptas[COLS_MODELO].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

km = KMeans(n_clusters=7, random_state=42, n_init=10)
grid_aptas["cluster"] = km.fit_predict(X_scaled)

grid["cluster"] = np.nan
grid.loc[grid_aptas.index, "cluster"] = grid_aptas["cluster"]

resumen = grid_aptas.groupby("cluster").agg(
    n_celdas=("cell_id", "count"),
    **{col: (col, "mean") for col in COLS_MODELO}
).round(3)
print("=== RESUMEN CLUSTERS k=7 (solo celdas aptas) ===")
print(resumen.to_string())

centroides = pd.DataFrame(
    scaler.inverse_transform(km.cluster_centers_),
    columns=COLS_MODELO
).round(3)
centroides.index.name = "cluster"
print("\n=== CENTROIDES ===")
print(centroides.to_string())

grid.to_file(GRID_PATH, driver="GPKG")
print("Guardado OK")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

PESOS = {
    "n_gasoducto":      0.25,
    "n_porcino":        0.25,
    "n_suelo":          0.15,
    "n_pendiente":      0.10,
    "n_carretera":      0.07,
    "n_carretera_dist": 0.05,
    "n_nucleos":        0.10,
    "n_natura":         0.03,
}
assert abs(sum(PESOS.values()) - 1.0) < 1e-9

grid["score"] = sum(grid[col] * peso for col, peso in PESOS.items())
grid["score"] = (grid["score"] * 100).round(2)

def clasificar(s):
    if s >= 65: return "ÓPTIMO"
    if s >= 50: return "BUENO"
    if s >= 35: return "MODERADO"
    return "NO APTO"

# 'categoria' ya viene de la celda de exclusión (APTA/EXCLUIDO)
mask_excluido = grid["categoria"] == "EXCLUIDO"
grid.loc[~mask_excluido, "categoria"] = grid.loc[~mask_excluido, "score"].apply(clasificar)
grid.loc[mask_excluido, "score"] = 0.0

print(grid["categoria"].value_counts())
print(f"\nCeldas aptas: {(grid['categoria'] != 'EXCLUIDO').sum():,}")

grid.to_file(GRID_PATH, driver="GPKG")
grid.drop(columns="geometry").to_csv(PROCESSED_DIR / "scoring_biometano.csv", index=False)
print("Guardado OK")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)

UMBRAL_BIOMASA   = 200_000
UMBRAL_GASODUCTO = 10_000
UMBRAL_PENDIENTE = 15
UMBRAL_CARRETERA = 2

# ── Viables (sobre celdas aptas, cluster y categoria ya son consistentes) ──
aptas = grid[grid["categoria"] != "EXCLUIDO"].copy()

mask_viable = (
    (aptas["biomasa_10km"]    >= UMBRAL_BIOMASA) &
    (aptas["dist_gasoducto"] <= UMBRAL_GASODUCTO) &
    (aptas["pendiente_media"] <= UMBRAL_PENDIENTE) &
    (aptas["pts_carretera"]  >= UMBRAL_CARRETERA)
)
viables = aptas[mask_viable].copy()

print(f"Viables: {len(viables):,}")

print("\n=== DISTRIBUCIÓN CLUSTERS EN VIABLES ===")
print(viables["cluster"].value_counts().sort_index())

print("\n=== SCORE EN VIABLES ===")
print(viables["score"].describe().round(1))

print("\n=== TOP 10 ===")
cols_show = ["cell_id", "cluster", "score", "biomasa_10km",
             "dist_gasoducto", "pendiente_media", "pts_carretera"]
print(viables.nlargest(10, "score")[cols_show].to_string(index=False))

viables.to_file(PROCESSED_DIR / "viables_interseccion.gpkg", driver="GPKG")
print(f"\nGuardado: {PROCESSED_DIR / 'viables_interseccion.gpkg'}")

In [ ]:
import geopandas as gpd
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")
GRID_PATH = PROCESSED_DIR / "grid_huesca_500m.gpkg"

grid = gpd.read_file(GRID_PATH)
grid_aptas = grid[grid["categoria"] != "EXCLUIDO"].copy()

resumen_clusters = grid_aptas.groupby("cluster").agg(
    n_celdas=("cell_id", "count"),
    score_medio=("score", "mean"),
    **{col: (col, "mean") for col in COLS_MODELO}
).round(2).sort_values("score_medio", ascending=False)

print(resumen_clusters.to_string())

In [ ]:
# ═══════════════════════════════════════════════════════
# 7. EXTRACCIÓN DEL TOP 10 REAL DE IDONEIDAD
# ═══════════════════════════════════════════════════════

# Filtrar para quedarnos solo con las celdas aptas (no excluidas)
grid_aptas = grid[grid["categoria"] != "EXCLUIDO"]

# Obtener las 10 con mayor puntuación
top10 = grid_aptas.nlargest(10, "score")

print("\n=== TOP 10 REAL DE IDONEIDAD (SITUADAS EN HUESCA) ===")
cols_top = ["cell_id", "score", "categoria", "dist_gasoducto", "biomasa_10km", "pendiente_media", "pts_suelo"]
print(top10[cols_top].to_string(index=False))

# Guardar el Top 10 en un archivo independiente por si quieres aislarlo en QGIS
from pathlib import Path
PROCESSED_DIR = Path("../../data/processed")
top10.to_file(PROCESSED_DIR / "top_10_localizaciones_biometano.gpkg", driver="GPKG")
print(f"\nGuardado: {PROCESSED_DIR / 'top_10_localizaciones_biometano.gpkg'} con las mejores parcelas.")

In [ ]:
score_viables = viables["score"].mean().round(1)
score_top10   = top10["score"].mean().round(1)

print(f"Score medio viables: {score_viables}")
print(f"Score medio top 10:  {score_top10}")

Score medio viables: 71.8
Score medio top 10:  85.2


In [ ]:
import geopandas as gpd
import pydeck as pdk
import pandas as pd
import numpy as np
from pathlib import Path

DELIM_DIR = Path("../../data/raw/delimitations")
PROCESSED_DIR = Path("../../data/processed")
MAP_DIR = Path("../../data/map/idoneidad_scoring_clustering")
MAP_DIR.mkdir(parents=True, exist_ok=True)

# ── Cargar datos ──────────────────────────────────────────────────
grid = gpd.read_file(PROCESSED_DIR / "grid_huesca_500m.gpkg").to_crs("EPSG:4326")
provincia = gpd.read_file(DELIM_DIR / "Huesca_Delimitacion.geojson").to_crs("EPSG:4326")

# IDs de viables e intersección (de memoria)
ids_viables = set(viables["cell_id"])
ids_top10   = set(top10["cell_id"])

# ── Score medios para leyenda ─────────────────────────────────────
score_viables = round(viables["score"].mean(), 1)
score_top10   = round(top10["score"].mean(), 1)

# ── Paleta continua verde→rojo (n colores según nº de clusters) ───
def paleta_score(n):
    """Devuelve n colores RGB interpolados de verde oscuro a rojo."""
    puntos = [
        (26,  150,  65),
        (82,  183,  74),
        (166, 217, 106),
        (253, 224,  92),
        (253, 174,  97),
        (240, 108,  50),
        (215,  25,  28),
    ]
    if n == 1:
        return [puntos[0]]
    result = []
    for i in range(n):
        t = i / (n - 1) * (len(puntos) - 1)
        lo, hi = int(t), min(int(t) + 1, len(puntos) - 1)
        frac = t - lo
        r = round(puntos[lo][0] + frac * (puntos[hi][0] - puntos[lo][0]))
        g = round(puntos[lo][1] + frac * (puntos[hi][1] - puntos[lo][1]))
        b = round(puntos[lo][2] + frac * (puntos[hi][2] - puntos[lo][2]))
        result.append((r, g, b))
    return result

# ── Calcular score medio y características por cluster ────────────
grid_aptas = grid[grid["categoria"] != "EXCLUIDO"].copy()

COLS_FEATURES = {
    "n_gasoducto":      ("gasoducto",  "lejos del gasoducto",   "cerca del gasoducto"),
    "n_porcino":        ("porcino",    "porcino escaso",         "porcino abundante"),
    "n_suelo":          ("suelo",      "suelo deficiente",       "buen suelo"),
    "n_pendiente":      ("pendiente",  "terreno escarpado",      "terreno llano"),
    "n_carretera":      ("carretera",  "carretera deficiente",   "buena carretera"),
    "n_carretera_dist": ("dist_carr",  "lejos de carretera",     "cerca de carretera"),
    "n_nucleos":        ("nucleos",    "alejado de nucleos",     "cerca de nucleos"),
}

UMBRAL_BAJO = 0.35
UMBRAL_ALTO = 0.75

def nivel_score(s):
    if s >= 65: return "Optimo"
    if s >= 60: return "Bueno"
    if s >= 55: return "Moderado"
    return "Bajo"

def nombre_cluster(row_resumen):
    """Genera un nombre descriptivo a partir del perfil de un cluster."""
    score = row_resumen["score_medio"]
    nivel = nivel_score(score)
    rasgos = []
    for col, (_, etiq_bajo, etiq_alto) in COLS_FEATURES.items():
        if col not in row_resumen.index:
            continue
        val = row_resumen[col]
        if val < UMBRAL_BAJO:
            rasgos.append(etiq_bajo)
        elif val > UMBRAL_ALTO:
            rasgos.append(etiq_alto)
    if rasgos:
        descripcion = " y ".join(rasgos[:2])
        return f"{nivel} — {descripcion}"
    return nivel

# Resumen por cluster
cols_resumen = ["score"] + [c for c in COLS_FEATURES if c in grid_aptas.columns]
resumen_clusters = (
    grid_aptas.groupby("cluster")[cols_resumen]
    .mean()
    .rename(columns={"score": "score_medio"})
    .round(3)
    .sort_values("score_medio", ascending=False)
)

# Asignar colores dinámicamente según posición en el ranking de score
n_clusters = len(resumen_clusters)
colores_rgb = paleta_score(n_clusters)

cluster_info = {}
for rank, (cid, row) in enumerate(resumen_clusters.iterrows()):
    rgb = colores_rgb[rank]
    hex_color = "#{:02x}{:02x}{:02x}".format(*rgb)
    cluster_info[int(cid)] = {
        "color":       list(rgb),
        "hex":         hex_color,
        "nombre":      nombre_cluster(row),
        "score_medio": round(row["score_medio"], 1),
    }

# ── Etiqueta carretera ────────────────────────────────────────────
label_carretera = {1: "Local", 2: "Secundaria", 3: "Principal", 4: "Autovia/Autopista"}

# ── Construir features ────────────────────────────────────────────
features = []
for _, row in grid.iterrows():
    geom = row.geometry
    if geom is None or geom.is_empty:
        continue

    cid = row["cell_id"]
    if row["categoria"] == "EXCLUIDO":
        color = [80, 80, 80, 100]
    elif cid in ids_top10:
        color = [128, 0, 200, 240]
    elif cid in ids_viables:
        color = [255, 105, 180, 200]
    else:
        k = row["cluster"]
        if pd.notna(k):
            info = cluster_info.get(int(k), {})
            color = info.get("color", [120, 120, 120]) + [160]
        else:
            color = [120, 120, 120, 80]

    pts_carr = int(row["pts_carretera"]) if pd.notna(row["pts_carretera"]) else 0

    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for p in polys:
        features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {
                "cell_id":        int(cid),
                "cluster":        int(row["cluster"]) if pd.notna(row["cluster"]) else -1,
                "score":          round(float(row["score"]), 1) if pd.notna(row["score"]) else 0,
                "biomasa":        int(row["biomasa_10km"]) if pd.notna(row["biomasa_10km"]) else 0,
                "gasoducto":      round(float(row["dist_gasoducto"]) / 1000, 1) if pd.notna(row["dist_gasoducto"]) else 0,
                "pendiente":      round(float(row["pendiente_media"]), 1) if pd.notna(row["pendiente_media"]) else 0,
                "carretera":      label_carretera.get(pts_carr, "Desconocida"),
                "carretera_dist": round(float(row["dist_carretera"]) / 1000, 1) if pd.notna(row["dist_carretera"]) else 0,
                "nucleos":        round(float(row["dist_nucleos"]) / 1000, 1) if pd.notna(row["dist_nucleos"]) else 0,
                "suelo":          int(row["pts_suelo"]) if pd.notna(row["pts_suelo"]) else 0,
                "color":          color,
                "motivo": row["motivo_exclusion"] if row["categoria"] == "EXCLUIDO" else "N/A",
            }
        })

geojson_grid = {"type": "FeatureCollection", "features": features}

# ── Límite provincia ──────────────────────────────────────────────
boundary_features = []
for geom in provincia.geometry:
    polys = [geom] if geom.geom_type == "Polygon" else list(geom.geoms)
    for p in polys:
        boundary_features.append({
            "type": "Feature",
            "geometry": {"type": "Polygon", "coordinates": [list(p.exterior.coords)]},
            "properties": {}
        })

# ── Layers ────────────────────────────────────────────────────────
layer_grid = pdk.Layer(
    "GeoJsonLayer",
    data=geojson_grid,
    stroked=True,
    filled=True,
    get_fill_color="properties.color",
    get_line_color=[0, 0, 0, 30],
    get_line_width=50,
    line_width_min_pixels=1,
    pickable=True,
)

layer_boundary = pdk.Layer(
    "GeoJsonLayer",
    data={"type": "FeatureCollection", "features": boundary_features},
    stroked=True,
    filled=False,
    get_line_color=[255, 255, 255, 200],
    get_line_width=800,
    line_width_min_pixels=2,
)

# ── Vista ─────────────────────────────────────────────────────────
view = pdk.ViewState(
    longitude=-0.08, latitude=41.95,
    zoom=8, min_zoom=7, max_zoom=13,
    pitch=0, bearing=0,
)

# ── Tooltip ───────────────────────────────────────────────────────
tooltip = {
    "html": """
        <b>Celda {cell_id}</b><br/>
        Cluster: {cluster}<br/>
        Score: {score}<br/>
        Biomasa 10km: {biomasa} plazas<br/>
        Dist. gasoducto: {gasoducto} km<br/>
        Dist. nucleos: {nucleos} km<br/>
        Tipo carretera: {carretera}<br/>
        Dist. carretera: {carretera_dist} km<br/>
        Pendiente media: {pendiente} grados<br/>
        Suelo: {suelo}<br/>
        Motivo exclusion: {motivo}
    """,
    "style": {"backgroundColor": "#111", "color": "white",
              "fontSize": "12px", "padding": "8px"},
}

deck = pdk.Deck(
    layers=[layer_grid, layer_boundary],
    initial_view_state=view,
    tooltip=tooltip,
    map_style="https://basemaps.cartocdn.com/gl/dark-matter-gl-style/style.json",
)

# ── Leyenda dinámica ──────────────────────────────────────────────
filas_clusters = ""
for cid, info in sorted(cluster_info.items(),
                        key=lambda x: x[1]["score_medio"], reverse=True):
    filas_clusters += (
        f'  <div><span style="display:inline-block;width:14px;height:14px;'
        f'background:{info["hex"]};border-radius:3px;margin-right:8px;'
        f'vertical-align:middle;"></span>'
        f'C{cid} â {info["nombre"]} ({info["score_medio"]})</div>\n'
    )

leyenda_html = f"""
<div style="position:absolute; bottom:30px; left:20px; background:rgba(0,0,0,0.75);
            padding:14px 18px; border-radius:8px; font-family:sans-serif;
            font-size:13px; color:white; z-index:999; line-height:1.9;">
  <div style="font-weight:bold; font-size:14px; margin-bottom:8px;">Idoneidad â Planta Biometano Â· Huesca</div>

  <div style="font-weight:bold; margin-top:10px; margin-bottom:4px; border-top:1px solid rgba(255,255,255,0.3); padding-top:8px;">Seleccion final</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#8000c8;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>Top 10 ambos modelos (score medio: {score_top10})</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#ff69b4;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>Viable en ambos modelos (score medio: {score_viables})</div>

  <div style="font-weight:bold; margin-top:10px; margin-bottom:4px; border-top:1px solid rgba(255,255,255,0.3); padding-top:8px;">Clusters K-Means</div>
{filas_clusters}
  <div style="font-weight:bold; margin-top:10px; margin-bottom:4px; border-top:1px solid rgba(255,255,255,0.3); padding-top:8px;">Zonas excluidas</div>
  <div><span style="display:inline-block;width:14px;height:14px;background:#505050;border-radius:3px;margin-right:8px;vertical-align:middle;"></span>Excluido (ver motivo en tooltip)</div>
</div>
"""

html_output = deck.to_html(as_string=True)
html_output = html_output.replace("</body>", leyenda_html + "</body>")

output_html = MAP_DIR / "mapa_final_biometano.html"
with open(output_html, "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"Guardado: {output_html}")


In [ ]:
from pathlib import Path
RAW_SUELO = Path("../../data/raw/06_clasificacion_suelo")
suelo = gpd.read_file(RAW_SUELO / "clasificacion_suelo_huesca.gpkg").to_crs("EPSG:25830")

top10_suelo = gpd.sjoin(
    top10[["cell_id", "score", "geometry"]],
    suelo[["clase", "uso_glob", "geometry"]],
    how="left",
    predicate="intersects"
)

print(top10_suelo[["cell_id", "score", "clase", "uso_glob"]].to_string(index=False))